# 🎬 Test đếm trên VIDEO THẬT — dây chuyền nhiều sản phẩm + query suite + lưu video

Nhiều video dây chuyền (kiện hàng, hệ thống chuyền đang chạy, dây chuyền rộng…),
bộ **query suite phong phú** (màu/phụ kiện/hành động/khó/tiếng Việt), người tách
**vạch (vào/ra) + vùng**, và **lưu video output** có vẽ vạch/vùng + box + số đếm.


## 1) Tải code + cài thư viện


In [ ]:
%cd /kaggle/working
!rm -rf VisionOS
!git clone -q https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git
%cd VisionOS/VisionOS
!git checkout -q claude/rebuild-visionos-codebase-tgg0mf
!git pull -q origin claude/rebuild-visionos-codebase-tgg0mf
!pip install -q ultralytics 'supervision>=0.21' opencv-python-headless


## 2) ✅ KIỂM TRA tải video (nhanh, không cần model)
Chạy trước để biết nguồn nào tải được. Video ❌ → báo mình ID để đổi nguồn.


In [ ]:
!python run_scenarios.py --download-only


## 3) Xem catalog + toàn bộ query suite


In [ ]:
!python run_scenarios.py --list
import recognition.video_catalog as vc
for t,groups in vc.QUERY_SUITES.items():
    print(f'\n=== {t.upper()} — {sum(len(v) for v in groups.values())} query ===')
    for g,qs in groups.items(): print(f'  [{g}] ' + ' | '.join(qs))


## ⭐ XEM TRƯỚC vạch/vùng trên MỌI video — LÀM TRƯỚC KHI ĐẾM
`--preview` vẽ **lưới %** + **vạch (vàng) / vùng (xanh)** lên frame CÓ vật của *mọi* video
(không cần model, chạy vài giây). Nhìn ảnh: nếu vạch/vùng đặt SAI chỗ vật đi qua →
báo mình toạ độ %, hoặc tự thử ngay `--only <video> --line 'x1,y1,x2,y2'` /
`--zone 'x1,y1;x2,y2;...'`. Đặt đúng vạch rồi mới đếm ra số.


In [ ]:
# Vẽ lưới % + vạch/vùng lên frame CÓ VẬT của MỌI video (không cần model, ~vài giây).
!python run_scenarios.py --preview /kaggle/working/prev

import glob, os
from IPython.display import Image, display, Markdown

# 🧩 Ảnh TỔNG HỢP — xem HẾT các trường hợp trong 1 ảnh (cuộn dọc).
allp = '/kaggle/working/prev/_ALL.jpg'
if os.path.exists(allp):
    display(Markdown('### 🧩 TỔNG HỢP tất cả trường hợp'))
    display(Image(filename=allp, width=900))

# Từng ảnh FULL-RES — đọc toạ độ theo lưới % để biết đặt vạch ở đâu.
for p in sorted(glob.glob('/kaggle/working/prev/*.jpg')):
    if os.path.basename(p) == '_ALL.jpg':
        continue
    display(Markdown(f'**{os.path.basename(p)}**  — vàng = vạch, xanh = vùng, lưới = %'))
    display(Image(filename=p, width=760))


## 🖊️ TỰ VẼ vạch/vùng bằng CHUỘT — 1 khung, nút ◀/▶ SANG ẢNH KHÁC
`draw_gallery([...])` cho **1 canvas** + nút **◀ Ảnh trước / Ảnh sau ▶** để chuyển qua
lại giữa các video (mỗi video giữ riêng nét vẽ). Cách vẽ (KHÔNG cần chọn chế độ):
- **VẠCH**: bấm 2 điểm → nút **➖ Xong VẠCH**.
- **VÙNG**: bấm ≥3 điểm → đóng bằng: bấm lại **điểm đầu** (vòng đỏ) / **bấm đúp** / nút **⬠ Xong VÙNG**.

Vẽ xong 1 ảnh bấm **Ảnh sau ▶** sang ảnh kế. Cuối cùng bấm **📋 Copy tất cả** rồi gửi
tôi (toạ độ đã gắn sẵn tên từng video), hoặc chạy CLI với `--only <video> --zone/--line`.

> ⚠️ Ô dưới **tự xoá cache module** để lấy code mới — nhưng vẫn phải chạy lại
> **cell "1) Tải code"** trước. Nếu vẫn lỗi: **Run → Restart kernel** rồi chạy lại từ cell 1.


In [ ]:
# Xoá cache module cũ để lấy CODE MỚI sau khi chạy lại cell 1 (khỏi Restart kernel).
import sys
for _m in [x for x in list(sys.modules) if x.startswith("recognition")]:
    del sys.modules[_m]

from recognition.draw_tool import draw_gallery
from IPython.display import HTML

# 1 khung + nút ◀/▶ để SANG ẢNH KHÁC. Sửa danh sách tuỳ ý (hoặc dùng "all" cho mọi video):
#   people: walk, subway, store, square · xe: 'giao lộ','cao tốc' · chuyền: milk, conv_pkg, conv_action…
HTML(draw_gallery(['subway', 'milk', 'walk', 'store', 'square', 'conv_pkg']))


## 4) 🚗 Đếm XE + lưu video output


In [ ]:
!python run_scenarios.py --task vehicles --max-frames 300 --save-dir /kaggle/working/scen_out


## 5) 🚶 Đếm NGƯỜI — cắt VẠCH (vào/ra) + đếm VÙNG + lưu video


In [ ]:
!python run_scenarios.py --task people --max-frames 300 --save-dir /kaggle/working/scen_out


## 6) 📦 Đếm chai trên chuyền (YOLO, nhanh) + lưu video


In [ ]:
!python run_scenarios.py --task conveyor --only milk --max-frames 300 --save-dir /kaggle/working/scen_out


## ⏳ Tải model 3B TRƯỚC khi đếm open-vocab (chạy 1 lần, CHỜ xong)
Các bài đếm open-vocab (mục dưới) dùng **LocateAnything-3B (~6GB)**. Lần đầu nạp mất
**3–8 phút** vì phải kéo model từ HuggingFace. Chạy cell này 1 lần để tải sẵn về cache
(có thanh tiến trình) — **ĐỪNG bấm Stop**. Sau đó các cell đếm chỉ mất ~1 phút nạp.

> `KeyboardInterrupt` khi đang tải = bạn/Kaggle đã **NGẮT giữa chừng**, KHÔNG phải lỗi
> code. Cứ chạy lại và để nó tải xong.


In [ ]:
# Tải sẵn model 3B về cache (~6GB, 3-8 phút lần đầu). ĐỪNG bấm Stop giữa chừng.
from huggingface_hub import snapshot_download
p = snapshot_download("nvidia/LocateAnything-3B")
print("✅ Model đã ở cache:", p)


## 7) 📦🧠 Đếm SẢN PHẨM dây chuyền + QUERY SUITE (open-vocab, ĐÃ tối ưu tốc độ)
`--suite` giờ là bản **LITE**: mỗi nhóm chỉ **1–2 query đại diện** (màu = đỏ/trắng…),
tự dùng **ít frame (60) + bỏ bớt frame (stride 2)** → chạy nhanh, hợp **Colab/Kaggle**
session ngắn. Trên Colab nên chạy **1 video** (`--only`).

- Nhanh hơn nữa: thêm `--suite-per-group 1` (mỗi nhóm 1 query) hoặc `--max-frames 40`.
- Cần bảng test lớn (chậm): đổi `--suite` → `--suite-full` (20–30+ query/bài).


In [ ]:
# Suite LITE (nhanh) trên 1 video kiện hàng — hợp Colab. Model 3B nạp ~1 phút.
!python run_scenarios.py --task conveyor --only pkg --suite --save-dir /kaggle/working/scen_out

# Nhanh nhất (1 query/nhóm, 40 frame):
# !python run_scenarios.py --task conveyor --only pkg --suite --suite-per-group 1 --max-frames 40
# Bảng test ĐẦY ĐỦ (chậm): đổi --suite -> --suite-full


In [ ]:
# Suite LITE cho NGƯỜI / XE (cũng dùng open-vocab). Chạy 1 video cho nhanh:
!python run_scenarios.py --task people --only walk --suite --save-dir /kaggle/working/scen_out
# !python run_scenarios.py --task vehicles --only 'cao tốc' --suite --save-dir /kaggle/working/scen_out


## 8) 🎥 Xem / tải video output


In [ ]:
import glob, os
from IPython.display import Video, display
vids = sorted(glob.glob('/kaggle/working/scen_out/**/*.mp4', recursive=True))
print(f'{len(vids)} video output:')
for p in vids: print('  ', p)
if vids:
    src = vids[0]; dst = '/kaggle/working/preview_h264.mp4'
    os.system(f'ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -pix_fmt yuv420p "{dst}"')
    print('Xem:', src); display(Video(dst, embed=True, width=700))


---
### Ghi chú
- **Cell 2** kiểm tra tải trước — video dây chuyền dùng endpoint tải chính thức của Pexels.
- `--suite`: nhiều trường hợp query phân nhóm (cột **nhóm** trong scorecard).
- Video output: vàng=vạch, xanh mờ=vùng, xanh dương=box+#id, banner số đếm.
- Thêm/đổi video hoặc query: sửa `recognition/video_catalog.py`.
- **Colab/session ngắn**: `--suite` đã tối ưu (lite + ít frame). Chạy 1 video với `--only`; muốn nhanh nhất thêm `--suite-per-group 1`.
